#Homework 5: Deploying Machine Learning Models

#Домашнее задание
> Примечание: иногда ваш ответ не полностью соответствует одному из вариантов. Это нормально. Выберите вариант, наиболее близкий к вашему решению. Если он находится ровно посередине между двумя вариантами, выберите большее значение.

В этом домашнем задании мы рекомендуем использовать Python 3.12 или 3.13.

В этом домашнем задании мы продолжим работу с набором данных для оценки лидов. Набор данных вам не понадобится: мы предоставим вам модель.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [6]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
!wget $data -O data-week-3.csv

--2025-10-26 07:53:09--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.05s   

2025-10-26 07:53:09 (19.5 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [7]:
df = pd.read_csv('data-week-3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [9]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [10]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, y_train)

    return dv, model

In [11]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [3]:
C = 1.0
n_splits = 5

In [13]:
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
  df_train = df_full_train.iloc[train_idx]
  df_val = df_full_train.iloc[val_idx]

  y_train = df_train.churn.values
  y_val = df_val.churn.values

  dv, model = train(df_train, y_train, C=C)
  y_pred = predict(df_val, dv, model)

  auc = roc_auc_score(y_val,y_pred)
  scores.append(auc)


print('C=%s %.3f +- %.3f' % (C,np.mean(scores), np.std(scores)))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

C=1.0 0.842 +- 0.007


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
scores

[np.float64(0.8446829053857807),
 np.float64(0.8453826479936432),
 np.float64(0.8333774834437085),
 np.float64(0.8347968759992326),
 np.float64(0.8517657117755952)]

In [15]:
dv , model = train(df_full_train,df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

np.float64(0.8584032088573997)

##Save the model

In [16]:
import pickle

In [17]:
output_file = f'model_C={C}.bin'
output_file

'model_C=1.0.bin'

In [18]:
f_out = open(output_file, 'wb')
pickle.dump((dv, model), f_out)
f_out.close()

In [19]:
with open(output_file, 'wb') as f_out:
  pickle.dump((dv, model), f_out)

##Load the mode

In [1]:
import pickle

In [4]:
model_file = f'model_C={C}.bin'

In [6]:
with open(model_file, 'rb') as f_in:
  dv, model = pickle.load(f_in)

In [7]:
dv, model

(DictVectorizer(sparse=False), LogisticRegression(max_iter=1000))

In [17]:
customer = {
    'gender': 'female',
    'seniorcitizen':0,
    'partner':'yes',
    'dependents':'no',
    'phoneservice':'no',
    'multiplelines':'no_phone_service',
    'internetservice':'dsl',
    'onlinesecurity':'no',
    'onlinebackup':'yes',
    'deviceprotection':'no',
    'techsupport':'no',
    'streamingtv':'no',
    'streamingmovies':'no',
    'contract':'month-to-month',
    'paperlessbilling':'yes',
    'paymentmethod':'electronic_check',
    'tenure':1,
    'monthlycharges':29.85,
    'totalcharges':29.85
}

In [18]:
X = dv.transform([customer])

In [19]:
model.predict_proba(X)[0, 1]

np.float64(0.6275953527536646)

##Вопрос 3

Давайте воспользуемся моделью!

* Написать скрипт для загрузки конвейера с помощью pickle
* Оцените этот рекорд:

``` python
{
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0
}
```


Какова вероятность того, что этот лид конвертируется?

* 0,333
* 0,533
* 0,733
* 0,933

Если при распаковке файлов возникают ошибки, проверьте их контрольную сумму:



```
$ md5sum pipeline_v1.bin
7d17d2e4dfbaf1e408e1a62e6e880d49 *pipeline_v1.bin
```



Мы подготовили конвейер со словарным векторизатором и моделью.

Обучение проводилось (примерно) с использованием следующего кода:

```
categorical = ['lead_source']
numeric = ['number_of_courses_viewed', 'annual_income']

df[categorical] = df[categorical].fillna('NA')
df[numeric] = df[numeric].fillna(0)

train_dict = df[categorical + numeric].to_dict(orient='records')

pipeline = make_pipeline(
    DictVectorizer(),
    LogisticRegression(solver='liblinear')
)

pipeline.fit(train_dict, y_train)
```

> Примечание : обучать модель не нужно. Этот код предназначен исключительно для справки.



In [30]:
data = 'https://github.com/DataTalksClub/machine-learning-zoomcamp/raw/refs/heads/master/cohorts/2025/05-deployment/pipeline_v1.bin'
!wget $data -O pipeline_v1.bin

--2025-10-26 08:51:07--  https://github.com/DataTalksClub/machine-learning-zoomcamp/raw/refs/heads/master/cohorts/2025/05-deployment/pipeline_v1.bin
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/master/cohorts/2025/05-deployment/pipeline_v1.bin [following]
--2025-10-26 08:51:07--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/refs/heads/master/cohorts/2025/05-deployment/pipeline_v1.bin
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1300 (1.3K) [application/octet-stream]
Saving to: ‘pipeline_v1.bin’

pipeline_v1.bin

In [31]:
import pickle

In [32]:
model_file = f'pipeline_v1.bin'

In [33]:
with open(model_file, 'rb') as f_in:
  dv, model = pickle.load(f_in)

In [34]:
dv, model

(DictVectorizer(), LogisticRegression(solver='liblinear'))

In [38]:
customer = {
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0
}

X = dv.transform([customer])
model.predict_proba(X)[0, 1]

np.float64(0.5336072702798061)

In [2]:
import requests

In [3]:
url = 'http://localhost:9696/predict'

In [6]:
client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
}

In [4]:
requests.post(url, json=client).json()

{'churn': True, 'churn_probability': 0.5340417283801275}

## Вопрос 6
Давайте запустим ваш Docker-контейнер!

После запуска оцените этого клиента еще раз:

``` 
url = "YOUR_URL"
client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0
}
requests.post(url, json=client).json()
```

Какова вероятность того, что этот лид конвертируется?

* 0,39
* 0,59
* 0,79
* 0,99

In [8]:
requests.post(url, json=client).json()

{'conversion_probability': 0.5340417283801275}